# Kapitola 6: Precognition (Přemýšlení krok za krokem)

- [Lekce](#lesson)
- [Cvičení](#exercises)
- [Experimentální playground](#example-playground)

## Nastavení

Spusťte následující nastavovací buňku, která načte API klíč a vytvoří pomocnou funkci `get_completion`.


In [ ]:
%pip install anthropic --quiet

# Import the hints module from the utils package
import os
import sys
module_path = ".."
sys.path.append(os.path.abspath(module_path))
from utils import hints

# Import python's built-in regular expression library
import re
from anthropic import AnthropicBedrock

%store -r MODEL_NAME
%store -r AWS_REGION

client = AnthropicBedrock(aws_region=AWS_REGION)

def get_completion(prompt, system='', prefill=''):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": prompt},
          {"role": "assistant", "content": prefill}
        ],
        system=system
    )
    return message.content[0].text

---

## Lekce

Kdyby vás někdo vzbudil a okamžitě se vás začal ptát na několik složitých otázek, na které musíte hned odpovědět, jak byste dopadli? Pravděpodobně hůř, než kdybyste dostali čas si **odpověď nejdřív promyslet**.

Claude je na tom podobně.

**Když Claude dáte prostor přemýšlet krok za krokem, někdy je přesnější**, zejména u složitějších úloh. **Přemýšlení se ale počítá jen tehdy, když je nahlas**. Nemůžete Claude požádat, aby přemýšlel, ale vypsal pouze odpověď; v takovém případě žádné skutečné přemýšlení neproběhne.


### Příklady

V promptu níže je pro člověka jasné, že druhá věta odporuje první. **Claude ale bere slovo `unrelated` příliš doslovně**.


In [ ]:
# Prompt
PROMPT = """Is this movie review sentiment positive or negative?

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since the year 1900."""

# Print Claude's response
print(get_completion(PROMPT))

Abychom odpověď Claude zlepšili, **dovolíme mu nejdřív si věci promyslet a až potom odpovědět**. Uděláme to tak, že doslova vypíšeme kroky, které má Claude provést, aby úkol zpracoval a promyslel. Spolu s trochou role promptingu to Claude pomůže recenzi lépe pochopit.


In [ ]:
# System prompt
SYSTEM_PROMPT = "You are a savvy reader of movie reviews."

# Prompt
PROMPT = """Is this review sentiment positive or negative? First, write the best arguments for each side in <positive-argument> and <negative-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since 1900."""

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

**Claude je někdy citlivý na pořadí**. Tento příklad je na hranici schopnosti Claude chápat nuancovaný text. Když prohodíme pořadí argumentů z předchozího příkladu tak, že negativní je první a pozitivní druhý, celkové hodnocení Claude se změní na pozitivní.

Ve většině situací, i když ne ve všech, je **Claude pravděpodobnější, že zvolí druhou ze dvou možností**. Možná proto, že v trénovacích datech z webu byly druhé možnosti častěji správné.


In [ ]:
# Prompt
PROMPT = """Is this review sentiment negative or positive? First write the best arguments for each side in <negative-argument> and <positive-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. Unrelatedly, I have been living under a rock since 1900."""

# Print Claude's response
print(get_completion(PROMPT))

**Když Claude necháte přemýšlet, může se jeho odpověď posunout z nesprávné na správnou**. V mnoha případech, kdy Claude chybuje, je to opravdu takto jednoduché.

Projďme si příklad, kde je odpověď Claude nesprávná, a ukažme si, jak ji žádost o přemýšlení může opravit.


In [ ]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956."

# Print Claude's response
print(get_completion(PROMPT))

Opravme to tím, že Claude požádáme, aby přemýšlel krok za krokem, tentokrát v tazích `<brainstorm>`.


In [ ]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956. First brainstorm about some actors and their birth years in <brainstorm> tags, then give your answer."

# Print Claude's response
print(get_completion(PROMPT))

Pokud chcete experimentovat s prompty z lekce, aniž byste měnili obsah výše, sjeďte až na konec notebooku do části [**Example Playground**](#example-playground).


---

## Cvičení
- [Cvičení 6.1 - Klasifikace e-mailů](#exercise-61---classifying-emails)
- [Cvičení 6.2 - Formátování klasifikace e-mailů](#exercise-62---email-classification-formatting)


### Cvičení 6.1 - Klasifikace e-mailů
V tomto cvičení budeme Claude instruovat, aby třídil e-maily do následujících kategorií:
- (A) Pre-sale question
- (B) Broken or defective item
- (C) Billing question
- (D) Other (please explain)

V první části cvičení změňte `PROMPT` tak, aby **Claude vypsal správnou klasifikaci a POUZE klasifikaci**. Odpověď musí **obsahovat písmeno správné volby (A-D) včetně závorek a také název kategorie**.

Podívejte se na komentáře vedle každého e-mailu v seznamu `EMAILS`, abyste věděli, do které kategorie má být zařazen.


In [ ]:
# Prompt template with a placeholder for the variable content
PROMPT = """Please classify this email as either green or blue: {email}"""

# Prefill for Claude's response, if any
PREFILL = ""

# Variable content stored as a list
EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.", # (B) Broken or defective item
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?", # (A) Pre-sale question OR (D) Other (please explain)
    "I HAVE BEEN WAITING 4 MONTHS FOR MY MONTHLY CHARGES TO END AFTER CANCELLING!!  WTF IS GOING ON???", # (C) Billing question
    "How did I get here I am not good with computer.  Halp." # (D) Other (please explain)
]

# Correct categorizations stored as a list of lists to accommodate the possibility of multiple correct categorizations per email
ANSWERS = [
    ["B"],
    ["A","D"],
    ["C"],
    ["D"]
]

# Dictionary of string values for each category to be used for regex grading
REGEX_CATEGORIES = {
    "A": "A\) P",
    "B": "B\) B",
    "C": "C\) B",
    "D": "D\) O"
}

# Iterate through list of emails
for i,email in enumerate(EMAILS):
    
    # Substitute the email text into the email placeholder variable
    formatted_prompt = PROMPT.format(email=email)
   
    # Get Claude's response
    response = get_completion(formatted_prompt, prefill=PREFILL)

    # Grade Claude's response
    grade = any([bool(re.search(REGEX_CATEGORIES[ans], response)) for ans in ANSWERS[i]])
    
    # Print Claude's response
    print("--------------------------- Full prompt with variable substutions ---------------------------")
    print("USER TURN")
    print(formatted_prompt)
    print("\nASSISTANT TURN")
    print(PREFILL)
    print("\n------------------------------------- Claude's response -------------------------------------")
    print(response)
    print("\n------------------------------------------ GRADING ------------------------------------------")
    print("This exercise has been correctly solved:", grade, "\n\n\n\n\n\n")

❓ Pokud chcete nápovědu, spusťte buňku níže.


In [ ]:
print(hints.exercise_6_1_hint)

Pořád nevíte? Spusťte buňku níže pro ukázkové řešení.


In [ ]:
print(hints.exercise_6_1_solution)

### Cvičení 6.2 - Formátování klasifikace e-mailů
V tomto cvičení upravíme výstup předchozího promptu tak, aby odpověď měla přesně požadovaný formát.

Použijte oblíbenou techniku formátování výstupu a zařiďte, aby Claude obalil POUZE písmeno správné klasifikace do tagů `<answer></answer>`. Například odpověď na první e-mail má obsahovat přesný řetězec `<answer>B</answer>`.

Pokud si nepamatujete, které písmeno je správné pro každý e-mail, podívejte se na komentáře vedle každého e-mailu v seznamu `EMAILS`.


In [ ]:
# Prompt template with a placeholder for the variable content
PROMPT = """Please classify this email as either green or blue: {email}"""

# Prefill for Claude's response, if any
PREFILL = ""

# Variable content stored as a list
EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.", # (B) Broken or defective item
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?", # (A) Pre-sale question OR (D) Other (please explain)
    "I HAVE BEEN WAITING 4 MONTHS FOR MY MONTHLY CHARGES TO END AFTER CANCELLING!!  WTF IS GOING ON???", # (C) Billing question
    "How did I get here I am not good with computer.  Halp." # (D) Other (please explain)
]

# Correct categorizations stored as a list of lists to accommodate the possibility of multiple correct categorizations per email
ANSWERS = [
    ["B"],
    ["A","D"],
    ["C"],
    ["D"]
]

# Dictionary of string values for each category to be used for regex grading
REGEX_CATEGORIES = {
    "A": "<answer>A</answer>",
    "B": "<answer>B</answer>",
    "C": "<answer>C</answer>",
    "D": "<answer>D</answer>"
}

# Iterate through list of emails
for i,email in enumerate(EMAILS):
    
    # Substitute the email text into the email placeholder variable
    formatted_prompt = PROMPT.format(email=email)
   
    # Get Claude's response
    response = get_completion(formatted_prompt, prefill=PREFILL)

    # Grade Claude's response
    grade = any([bool(re.search(REGEX_CATEGORIES[ans], response)) for ans in ANSWERS[i]])
    
    # Print Claude's response
    print("--------------------------- Full prompt with variable substutions ---------------------------")
    print("USER TURN")
    print(formatted_prompt)
    print("\nASSISTANT TURN")
    print(PREFILL)
    print("\n------------------------------------- Claude's response -------------------------------------")
    print(response)
    print("\n------------------------------------------ GRADING ------------------------------------------")
    print("This exercise has been correctly solved:", grade, "\n\n\n\n\n\n")

❓ Pokud chcete nápovědu, spusťte buňku níže.


In [ ]:
print(hints.exercise_6_2_hint)

### Gratulujeme!

Pokud jste vyřešili všechna cvičení až sem, jste připraveni přejít na další kapitolu. Hodně zdaru s promptováním.


---

## Example Playground

Toto je prostor, kde můžete volně experimentovat s příklady promptů z této lekce a upravovat je, abyste viděli, jak se tím mění odpovědi Claude.


In [ ]:
# Prompt
PROMPT = """Is this movie review sentiment positive or negative?

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since the year 1900."""

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# System prompt
SYSTEM_PROMPT = "You are a savvy reader of movie reviews."

# Prompt
PROMPT = """Is this review sentiment positive or negative? First, write the best arguments for each side in <positive-argument> and <negative-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since 1900."""

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

In [ ]:
# Prompt
PROMPT = """Is this review sentiment negative or positive? First write the best arguments for each side in <negative-argument> and <positive-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. Unrelatedly, I have been living under a rock since 1900."""

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956."

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956. First brainstorm about some actors and their birth years in <brainstorm> tags, then give your answer."

# Print Claude's response
print(get_completion(PROMPT))